# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [ ]:
# 1. Cấu hình
import json
import os
import uuid

from kaggle_secrets import UserSecretsClient

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"
BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"
RUNTIME_YAML_CONFIG = "train/runtime_Uni-MuMER-train.yaml"
YAML_CONFIG = BASE_YAML_CONFIG
NOTEBOOK_PATH = "uni-mumer-kaggle-dagshub v4.ipynb"
OUTPUT_DIR = "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
RUN_UUID = uuid.uuid4().hex
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
if not DAGSHUB_TOKEN:
    raise RuntimeError("Kaggle Secret DAGSHUB_TOKEN is missing")

os.environ.pop("PYTHONPATH", None)
os.environ.update({
    "PROJECT_DIR": PROJECT_DIR,
    "CONDA_DIR": CONDA_DIR,
    "ENV_DIR": ENV_DIR,
    "PYTHON": PYTHON,
    "BASE_YAML_CONFIG": BASE_YAML_CONFIG,
    "RUNTIME_YAML_CONFIG": RUNTIME_YAML_CONFIG,
    "YAML_CONFIG": YAML_CONFIG,
    "NOTEBOOK_PATH": NOTEBOOK_PATH,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_UUID": RUN_UUID,
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MLFLOW_FLATTEN_PARAMS": "TRUE",
    "MLFLOW_TAGS": json.dumps({
        "run_uuid": RUN_UUID,
        "source": "kaggle",
        "task": "sft",
        "dataset": "parquet_crohme_train",
    }),
})

print(f"Run UUID: {RUN_UUID}")
print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")

In [ ]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

In [ ]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

In [ ]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python --version
python -m pip install -q -r requirements.txt
python -m pip install -q -e train/LLaMA-Factory
python -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

In [ ]:
# 5. Runtime YAML Override (tùy chọn)
from scripts.runtime_yaml import prepare_runtime_yaml

USE_RUNTIME_YAML_OVERRIDE = True

YAML_OVERRIDES = {
    "dataset": "parquet_crohme_train",
    "max_samples": 2,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 64,
    "learning_rate": 1.0e-4,
    "lora_rank": 64,
    "logging_steps": 1,
    "save_steps": 20,
    "output_dir": "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora",
    
    # Để None nếu không muốn override
    "cutoff_len": None,
    "val_size": None,
    "per_device_eval_batch_size": None,
    "eval_steps": None,
    "save_total_limit": None,
    "lora_alpha": None,
    "lora_dropout": None,
    "warmup_ratio": None,
    "quantization_bit": None,
    "preprocessing_num_workers": None,
    "dataloader_num_workers": None,
    "bf16": None,
    "fp16": None,
}

YAML_CONFIG, OUTPUT_DIR, YAML_DATA = prepare_runtime_yaml(
    project_dir=PROJECT_DIR,
    base_yaml_config=BASE_YAML_CONFIG,
    runtime_yaml_config=RUNTIME_YAML_CONFIG,
    use_override=USE_RUNTIME_YAML_OVERRIDE,
    overrides=YAML_OVERRIDES,
    strict_keys=True,
)

# Cập nhật biến môi trường
mlflow_tags = json.loads(os.environ["MLFLOW_TAGS"])
mlflow_tags.update({
    "dataset": str(YAML_DATA.get("dataset", "")),
    "yaml_config": YAML_CONFIG,
    "yaml_override": str(USE_RUNTIME_YAML_OVERRIDE).lower(),
})

os.environ.update({
    "YAML_CONFIG": YAML_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "MLFLOW_TAGS": json.dumps(mlflow_tags),
})

In [ ]:
%%bash
# 6. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

In [ ]:
%%bash
# 7. Training
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

echo "Python: $(command -v python)"
echo "Torchrun: $(command -v torchrun)"
MPLBACKEND=Agg llamafactory-cli train "$YAML_CONFIG" "run_name=uni-mumer-${RUN_UUID:0:8}"

In [ ]:
%%bash
# 8. Upload và xác minh artifact
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --notebook "$NOTEBOOK_PATH"